# IT_TF Organism Class

## Brief Description
The following code contains the Organism class, including its constructor and methods. Paired with the population class, these two will act as the backbone for the IT_TF Genetic Algorithm

Documentation is a work in progress

In [14]:
import random
import numpy as np
from Bio import SeqIO, motifs
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.motifs import Motif

'''
Object organism

'''

class Organism():
    """ 
    Organism object for IT_TF Genetic Algorithm
    
    Information in Transcription Factor Binding Sites (TFBS) is typically conceived
    as being conveyed column-wise, measured through Information Content (IC).
    Information, however, can also be conveyed pairwise or between columns. Mutual 
    Information measures the information shared between two variables or columns. 
    
    The IT_TF project seeks to explore questions related to evolutionary preferences
    between these two ways of storing information and what this reveals about MI.
    This algorithm attempts to provide insight into this question by 
    modeling evolution in TFBS, fitting populations of organisms 
    based on Total Information (The sum of IC and MI).
    
    The algorithm uses the following approach (To be continued):
    
        1. Constructing Organism objects as a collection of randomly (weighted by %GC) generated
        TFBS
        2. Computing Total Information conveyed by the list of TFBS for each organism
        3. Generating mutations for each organism based on the organism's mutation rate
        and the poisson distribution, abiding by GC content.
        4. (To be continued but basically - Recombination operator, population development, and organism-organism competition)
    
       
    Parameters:
        N - Number of binding sites
        L - length of binding sites
        GC_cont - GC content of the organism's genome 
        Mut_rate - Probability of a base pair mutating
            Note: Currently used as the probability that a base could change into any base
            in the organism's alphabet, meaning 'mutations' that do not change 
            the base are possible. 
        Pseudocount - TBA
        MI_mode - Choice between computing MI based on adjacent pairs or overlapping pairs (In development)
        readin - Creates organism object with a list of TFBS read from a fasta file,
        overwriting N and L parameters
            Default: None
    """
    
    def __init__(self, N, L, GC_cont, mut_rate, Pseudocount = None, MI_mode = None, read_in = None):
        
        #Intiailizes received parameters
        self.number_bs = N
        self.length_bs = L
        self.mut_rate = mut_rate
        self.GC = GC_cont
        self.read_in = read_in
        
        #Initializes to be set parameters
        self.TFBS = []
        
        #Declares the organism's alphabet (Possible base pairs for a TFBS)
        self.alphabet = ["G","C","A","T"]
        
        #Declares weights for each letter based on GC
        weight_base = [(self.GC)/2, (self.GC/2), ((1-self.GC)/2), ((1-self.GC)/2)]
        
        print(weight_base)
        #If read_in is none
            #Initializes each TFBS as a random (albeit weighted by Genome GC) assortment of bases
        #Otherwise
            #Initializes array of TFBS from fasta file
        if self.read_in == None:
        
            #Initializes list of TFBS as a random assortment of bases weighted by %GC
            for i in range(self.number_bs):
                self.TFBS.append(''.join(random.choices(self.alphabet,weight_base,k = self.length_bs)))
        
        else:
            #Parses inputted file
            fas_iterator = SeqIO.parse(read_in,"fasta")
            
            #Adds each TFBS in file to the list
            for bs in fas_iterator:
                self.TFBS.append(str(bs.seq))
            
            #Reinitializes parameters
            self.number_bs = len(self.TFBS)
            self.length_bs = len(self.TFBS[0])
                
        #Initializes TI property 
        self.Compute_TI()       
        

    def Compute_TI(self):
        '''
        Organism method used to compute total information stored in an organism's
        list of TFBS. Total Information sums both Information Content and Mutual Information 
        of two columns. In this algorithm, fitting for Total Information allows us to explore 
        how TFBS prefer one manner of storing information over another.
        
        This method calculates TI of an organism by applying the following mathematical 
        simplification to the TI formula
        
        TI = IC_x + IC_y + MI_xy = 
        (Hg - Hx) + (Hg - Hy) + (Hx + Hy - Hxy) = 2Hg - Hxy
        
        IC_x = Information Content of column X
        IC_y = Information Content of column Y
        MI_xy = Information shared between column x and y
        
        Hg = Uncertainty in column X/Y BEFORE receiving information about the binding preferences of the TF
            Note: Based on an equal probability for each state of the variable (25% A, 25% T, 25% G, 25% C)
            
        Hx = Uncertainy in column X AFTER receiving information about the 
        binding preferences of the TF at this position of the motif. 
        
        Hy = Uncertainy in column X AFTER receiving information about the 
        binding preferences of the TF at this position of the motif.
        
        Hxy = Joint uncertainty of column X and Y 
            Note: The more biased this distribution is the more mutual information X and Y share.
            
        
        This method applies the stated equation by storing Hg as a constant, initializing
        a dictionary with 16 possible dinucleotide keys, calculating the frequency of each
        dinucleotide present in a pair of adjacent columns, and computing joint entropy (Hxy). 
        By repeating this process with every adjacent pair of columns (no column is double counted), 
        a cumulative joint entropy is computed. The below equation is then used to compute TI.
        
        TI_total = (2Hg - Hab) + (2Hg - Hcd) + (2Hg - Hef) = 6Hg - Hab - Hcd - Hef
        
        This method is ran in the constructor to initialize the TI parameter for every
        organism object. This method also initializes the joint entropy parameter
        for simple access by the Compute_MI method when/if it is called.
        '''
        
        #Initializes Hg as a constant representing the uncertainity of a variable before receiving information from the motif
        Hg = 2
        
        #Initializes Hxy
        Hxy = 0
        
        #Initializes a dictionary meant to hold the frequencies of each base pair in the first position
        #Placeholder: Calculates the TI of the first sequence's pair
        frequency_dic = {}
        
        #Populates frequency_dic based on all possible permutations of dinucleotides
        for element_outer in self.alphabet:
            for element_inner in self.alphabet:
                frequency_dic[element_inner + element_outer] = 0
        
        #Iterates through independent, adjacent pairs
        for i in range(0,self.length_bs-1,2):
            
            #Fills frequency array with dinucleotide counts
            for binding_site in self.TFBS:
                frequency_dic[binding_site[i] + binding_site[i+1]] += 1
            
            #Converts counts to frequencies
            for key in frequency_dic.keys():
                
                bp_prob = frequency_dic[key] / self.number_bs
                
                #Disregards if frequency is zero
                if bp_prob != 0:
                    
                    #Calculates joint entropy            
                    Hxy += (-1 * (bp_prob * np.log2(bp_prob)))
            
                #Sets the count to zero for the next loop
                frequency_dic[key] = 0
                
        #Sets the object's TI property to the computed value
        self.TI = (self.length_bs * Hg) - Hxy
        
        #Sets the object's cum_joint_entropy property to the sum of joint entropy between all adjacent columns
        #Assists Compute_MI method
        self.cum_joint_entropy = Hxy
        
    def Compute_IC(self):
        '''
        Organism method used to compute Information Content of the organism's TFBS.
        By calculating Information Content and Mutual Information over many generations
        of the population, the preferences of Transcription Factor Binding Sites towards
        one manner of storing information can be explored
        
        This method calculates column-wise information by iterating through the 
        list of TFBS by column, computing the frequency of each base at a column,
        and calculating shannon entropy given the computed frequencies at each column. 
        
        IC_x = Hg - Hx = 2 - Hx
        '''
        #Variable to store cumulative IC
        self.IC = 0
        
        #Variable to store shannon entropy
        Hx = 0 
        
        #Variables to store counts
        G_count = 0
        C_count = 0
        A_count = 0
        T_count = 0
        
        #Iterates through list of TFBS to get column wise frequencies
        
        #Iterates through column to get all bases at each column
        for col in range(self.length_bs):
            
            #Iterates through row to get all bases at a column
            for row in range(self.number_bs):
                
                #Accesses 2D array elements in a certain column
                bp_TFBS = self.TFBS[row][col]
                
                #Keeps count of the base 
                if bp_TFBS == 'G':
                    G_count += 1
                    
                elif bp_TFBS == 'C':
                    C_count += 1
                
                elif bp_TFBS == 'A':
                    A_count += 1
                    
                else:
                    T_count += 1
                    
            #After counting all the bases in a certain column, calculates frequency
            G_count /= self.number_bs
            C_count /= self.number_bs
            A_count /= self.number_bs
            T_count /= self.number_bs
            
            print(G_count,C_count,A_count,T_count)
            #Calculates uncertainty of the column after knowing the frequencies of each base
            if G_count != 0:
                Hx += -(G_count * np.log2(G_count)) 
            
            if C_count != 0:
                Hx += -(C_count*np.log2(C_count))
            
            if A_count != 0:
                Hx += -(A_count*np.log2(A_count))
                
            if T_count != 0:
                Hx += -(T_count*np.log2(T_count))
            
            #Calculates Information Content of the column and adds it to cumulative IC variable
            self.IC += 2 - Hx
            
            #Sets variables back to zero
            Hx = 0
            
                
                    
                
        
    def Compute_MI(self):
        '''
        Organism method used to compute Mutual Information of the organism's TFBS.
        By calculating Information Content and Mutual Information over many generations
        of the population, the preferences of Transcription Factor Binding Sites towards
        one manner of storing information can be explored
        
        This method takes advantage of the prior computation performed by 
        Compute_TI to calculate joint entropy across all adjacent pairs (no overlap)
        and Compute_IC to calculate uncertainty of each column. This method then calculates
        MI as the difference between cumulative uncertainty and cumulative joint entropy
        
        MI = (Ha + Hb - Hab) + (Hc + Hd - Hcd) + (He + Hf - Hef) = 
        (Ha + Hb + Hc + Hd + He + Hf) - (Hab + Hcd + Hef)
        
        Cumulative uncertainty - Cumulative joint entropy
        
        '''
        
        #Converts IC back to uncertainty
        cum_entropy = -(self.IC) + 2 * self.length_bs
        
        #Calculates MI
        self.MI = cum_entropy - self.cum_joint_entropy
            
        

    def Mutate(self):
        '''
        Organism method used to apply mutations at random positions of the organism's 
        genome (The list of TFBS). In order to investigate information conveyed
        column-wise and pair-wise, the ability of information (both MI and IC)
        to withstand mutation must also be explored. This method provides the means
        for mutation to permit this exploration and introduces genetic diversity within 
        the population so as to permit these populations to scale the fitness landscape.
        
        The method uses the mutation rate and genome length to calculate an expected 
        number of mutations, referencing the poisson distribution to get a number 
        of point mutations occuring at this round of mutation for an organism. 
        The number of mutations is used to initialize a list of length (num_of_mut) containing unique 
        positions within the genome for mutation. Iterating through the list,
        the base pair at each position is mutated using a weight based on GC content.
        
        Note: Currently, mutations that mutate a base pair to the same base pair are enabled (Ex: G --> G)
        '''
        
        #Calculates expected value as mt rate * length of sequence * number of mutations
        expected_mut = self.mut_rate * self.number_bs * self.length_bs
        
        #Generates the # of mutations based on expected value using the poisson distribution
        num_mut = np.random.poisson(expected_mut)
        
        #Indicates cutoff value for each base pair
        g_freq = self.GC/2
        c_freq = self.GC
        a_freq = self.GC + 0.5 * (1-self.GC)
        
        #Prepares to draw num_mut amount of numbers between 1 and the genome length (sites for mutation)
        collection_of_positions = []
        sufficient_mut = 0
        
        #Iterative loop to ensure num_mut of unique mutations 
        while sufficient_mut < num_mut: 
            temp_pos = random.randrange(0,(self.number_bs*self.length_bs) - 1)
            
            #Note: Look for function that sorts and checks progressive elements
            if not(temp_pos in collection_of_positions):
                collection_of_positions.append(temp_pos)
                sufficient_mut += 1
        
        #Iterates through list and finds referenced position within TFBS
        for pos in collection_of_positions:
            
            #Locates position within list of TFBS
            row = pos // self.length_bs
            col = pos % self.length_bs
            
            #Generates random float between 0 and 1 used to mutate the positions stored in the above array
            #Mutations weighted by GC content
            rand_float = random.random()
            
            #Applies mutation
            if rand_float < g_freq:
                self.TFBS[row] = self.TFBS[row][:col] + 'G' + self.TFBS[row][(col+1):]
            
            elif rand_float < c_freq:
                self.TFBS[row] = self.TFBS[row][:col] + 'C' + self.TFBS[row][(col+1):]
            
            elif rand_float < a_freq:
                self.TFBS[row] = self.TFBS[row][:col] + 'A' + self.TFBS[row][(col+1):]
            
            else:
                self.TFBS[row] = self.TFBS[row][:col] + 'T' + self.TFBS[row][(col+1):]        
        
    def Export_Fasta(self,file_name):
        '''
        After the object's constructor is called, the following method can be used
        to export the array of TFBS into a fasta file
        
        Converts array of TFBS into an array of Seq Objects and writes Seq Objects into fasta file. 
        
        Paramaters
        ----------
        file_name - Str - Name of the fasta file to write to 

        Returns
        -------

        '''
        
        #Initializes Seq Object
        seq_list = []
        
        #Creates list of seq objects with sequences corresponding to the organism's TFBS
        for bs in self.TFBS:
            seq_list.append(SeqRecord(Seq(bs),id = ""))
            
        #Writes fasta file if file_name is valid fasta file name.
        if ((file_name[-3:] == 'fas') or (file_name[-5:] == 'fasta')):
            SeqIO.write(seq_list,file_name,'fasta')
        
        else:
            raise(Exception("Please enter a valid file to export to"))
    
    def Generate_Logo(self,file_name):
        '''
        Method to generate logo of organism's TFBS 
        
        Parameters
        -----------
        file_name - Str - Name of the file to be output to (Must be png)
        
        Returns
        -------
        None
        
        '''
        #Initializes Seq Object
        seq_list = []
        
        #Creates list of seq objects with sequences corresponding to the organism's TFBS
        for bs in self.TFBS:
            seq_list.append(Seq(bs))
            
        #Creates motif and generates logo
        object_motif = motifs.create(seq_list)
        
        object_motif.weblogo(file_name, format = 'PNG')
        


The following code blocks are used to test the TI, IC, and MI methods

In [15]:
#Generates an organism with perfectly conserved TFBS
organism1 = organism1 = Organism(100,10,GC_cont = 0.50, mut_rate= 0.05,read_in = "../TI_Test/perfect_conservation.fas")

#Calls methods
 
organism1.Compute_IC()
organism1.Compute_MI()

print("Perfectly conserved TFBS: ")
print("IC (Sum of ICs of each column | Total IC):",organism1.IC/organism1.length_bs,organism1.IC)
print("MI (Sum of MIs of each pair of columns):", organism1.MI/10,organism1.MI)
print("TI (Sum of TIs of each pair of columns):",organism1.TI/10,organism1.TI)

[0.25, 0.25, 0.25, 0.25]
0.0 0.0 1.0 0.0
0.0 0.0 1.001 0.0
0.0 0.0 1.001001 0.0
0.0 0.0 1.001001001 0.0
0.0 0.0 1.001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
0.0 0.0 1.001001001001001 0.0
Perfectly conserved TFBS: 
IC (Sum of ICs of each column | Total IC): 2.0013725462933127 40.027450925866255
MI (Sum of MIs of each pair of columns): -0.002745092586625475 -0.02745092586625475
TI (Sum of TIs of each pair of columns): 4.0 40.0
